# End-to-End Machine Learning Pipeline on Tesla Sales/Price Data

This notebook demonstrates:

- Data Loading
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Regression Modeling
- Hyperparameter Tuning
- Model Evaluation
- Time Series Forecasting
- Future Predictions


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')
df.head()


In [ ]:
df.shape, df.info()

## Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Avg_Price_USD'], kde=True)
plt.title('Price Distribution')
plt.show()


In [ ]:
corr_cols = ['Estimated_Deliveries','Production_Units','Avg_Price_USD',
             'Battery_Capacity_kWh','Range_km','CO2_Saved_tons','Charging_Stations']

plt.figure(figsize=(8,6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm')
plt.show()


## Feature Engineering

In [ ]:
df['Date'] = pd.to_datetime(df[['Year','Month']].assign(DAY=1))

df['Delivery_Production_Ratio'] = (
    df['Estimated_Deliveries'] / df['Production_Units']
)

df.head()


## Regression Model: Predict Tesla Average Price

In [ ]:
X = df.drop(columns=['Avg_Price_USD','Date'])
y = df['Avg_Price_USD']

categorical_cols = X.select_dtypes(include='object').columns
numeric_cols = X.select_dtypes(exclude='object').columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("R2  :", r2_score(y_test, preds))


## Hyperparameter Tuning

In [ ]:
param_grid = {
    'regressor__n_estimators':[100,200],
    'regressor__max_depth':[10,20,None]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score :", grid.best_score_)


## Time Series Forecasting (Monthly Deliveries)

In [ ]:
monthly = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()

monthly.head()


In [ ]:
from sklearn.linear_model import LinearRegression

monthly['Time_Index'] = np.arange(len(monthly))

ts_model = LinearRegression()
ts_model.fit(monthly[['Time_Index']], monthly['Estimated_Deliveries'])

monthly['Forecast'] = ts_model.predict(monthly[['Time_Index']])

plt.figure(figsize=(10,5))
plt.plot(monthly['Date'], monthly['Estimated_Deliveries'], label='Actual')
plt.plot(monthly['Date'], monthly['Forecast'], label='Trend Forecast')
plt.legend()
plt.title('Tesla Deliveries Forecast')
plt.show()


In [ ]:
future = pd.DataFrame({
    'Time_Index': np.arange(
        len(monthly),
        len(monthly)+12
    )
})

future['Forecast_Deliveries'] = ts_model.predict(future[['Time_Index']])

future


## Project Summary

This project covers:

1. Data preprocessing
2. EDA
3. Feature engineering
4. Regression modeling
5. Hyperparameter tuning
6. Time series forecasting
7. Business insights generation

Suitable for Data Analyst, Data Science, and Machine Learning portfolios.
